# Análisis Ventas Ecovida 2023-2025

**Empresa:** Ecovida (Alimentos Claudet)
**Período:** 2023-01-01 → 2025-01-14
**Fuente:** Sistema ERP Bsoft — Dataset de ventas transaccional

---

## Preguntas de Negocio

1. ¿Cuál es la contribución de ingresos y margen por canal de venta y cómo evolucionó en el período?
2. ¿Qué productos generan mayor rentabilidad versus volumen y existen oportunidades de mix de productos?
3. ¿Cuál es el desempeño de cada KAM y cómo impacta en la retención y crecimiento de clientes?
4. ¿Cómo optimizar la estrategia comercial considerando la alta concentración en Walmart y Cencosud?

---

## ⚙️ Configuración y Carga de Datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo visual consistente
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

# Carga del dataset
CSV_PATH = r"C:\Users\nick_\claude\ecovida\POWERBI\POWERBI\PANEL DE VENTAS\venta por canal.csv"

def limpiar_numerico(serie):
    return pd.to_numeric(
        serie.astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce"
    )

df = pd.read_csv(CSV_PATH, sep=";", encoding="latin-1", low_memory=False)

# Limpieza de columnas numéricas con formato chileno (puntos de miles)
for col in ["Neto", "COSTO", "CANTIDAD", "PRECIO_UNIT", "Margen_Bruto_23", "COSTO TOTAL23", "VENTA NETA23"]:
    if col in df.columns:
        df[col] = limpiar_numerico(df[col])

df["FECHA"] = pd.to_datetime(df["FECHA"], dayfirst=True, errors="coerce")
df["AÑO"]   = df["FECHA"].dt.year
df["MES"]   = df["FECHA"].dt.to_period("M")

# Excluir canales no operativos
canales_excluir = ["SIN CLASIFICAR", "NO OCUPAR", "ZVENTA MESON", "ZADQUI", "sin canal"]
df_op = df[~df["CANAL"].isin(canales_excluir)].copy()

print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Período: {df['FECHA'].min().date()} → {df['FECHA'].max().date()}")
print(f"Canales operativos: {df_op['CANAL'].unique().tolist()}")


## 1. Contexto de Negocio y Exploración de Datos

Ecovida, línea de productos de Alimentos Claudet, representa un portafolio de alimentos saludables con presencia en múltiples canales de distribución. Esta sección establece el volumen total de ventas, la distribución temporal y la cantidad de unidades comercializadas, proporcionando a los stakeholders una visión clara del alcance y tendencia del negocio en el período analizado.

In [ ]:
# Sección 1: Contexto de Negocio y Exploración de Datos

# 1.1 Cálculo de métricas agregadas globales
venta_total_millones = df['VENTA NETA23'].sum() / 1_000_000
cantidad_total_millones = df['CANTIDAD'].sum() / 1_000_000
num_registros = len(df)
fecha_inicio = df['FECHA'].min()
fecha_fin = df['FECHA'].max()

print(f"📊 Dataset: {num_registros:,} registros desde {fecha_inicio.date()} hasta {fecha_fin.date()}")
print(f"💰 Venta neta total: ${venta_total_millones:,.1f} millones CLP")
print(f"📦 Cantidad total: {cantidad_total_millones:,.1f} millones de unidades")

# 1.2 Preparación de datos para visualización temporal
df_temporal = df.groupby('FECHA').agg({
    'VENTA NETA23': 'sum',
    'CANTIDAD': 'sum'
}).reset_index().sort_values('FECHA')

# 1.3 Cálculo de tendencia (pendiente simple)
df_temporal['dias_desde_inicio'] = (df_temporal['FECHA'] - df_temporal['FECHA'].min()).dt.days
if len(df_temporal) > 1:
    pendiente_ventas = (df_temporal['VENTA NETA23'].iloc[-1] - df_temporal['VENTA NETA23'].iloc[0]) / len(df_temporal)
    tendencia = "creciente" if pendiente_ventas > 0 else "decreciente" if pendiente_ventas < 0 else "estable"
else:
    tendencia = "sin datos suficientes"

# 1.4 Visualización: Evolución temporal de ventas
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(df_temporal['FECHA'], df_temporal['VENTA NETA23']/1_000_000, 
         color='#2ecc71', linewidth=2.5, label='Venta Neta (Millones CLP)', marker='o', markersize=4, alpha=0.8)
ax1.set_xlabel('Fecha', fontsize=11, fontweight='bold')
ax1.set_ylabel('Venta Neta (Millones CLP)', fontsize=11, fontweight='bold', color='#2ecc71')
ax1.tick_params(axis='y', labelcolor='#2ecc71')
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.set_title('Ecovida: Evolución Temporal de Ventas Netas', fontsize=13, fontweight='bold', pad=20)

# Eje secundario para cantidad
ax2 = ax1.twinx()
ax2.plot(df_temporal['FECHA'], df_temporal['CANTIDAD']/1_000_000, 
         color='#e74c3c', linewidth=2.5, label='Cantidad (Millones de unidades)', marker='s', markersize=4, alpha=0.8)
ax2.set_ylabel('Cantidad (Millones de unidades)', fontsize=11, fontweight='bold', color='#e74c3c')
ax2.tick_params(axis='y', labelcolor='#e74c3c')

# Leyenda combinada
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10, framealpha=0.95)

plt.tight_layout()
plt.savefig("img/grafico_1.png", bbox_inches="tight", dpi=300)
plt.show()

# 1.5 Hallazgo principal
print(f"\n✅ HALLAZGO PRINCIPAL: Las ventas totales de Ecovida alcanzaron ${venta_total_millones:,.1f} mil millones CLP en el período analizado, con una tendencia {tendencia} y {cantidad_total_millones:,.1f} millones de unidades vendidas.")

## 2. Contribución de Ingresos y Margen por Canal de Venta

Esta sección analiza cómo cada canal de venta contribuye tanto en términos de ingresos netos como de margen bruto, identificando qué canales generan mayor volumen versus mayor rentabilidad. Comprender esta dinámica es crucial para optimizar la asignación de recursos y desarrollar estrategias diferenciadas por canal, especialmente en contextos donde canales pequeños pueden tener márgenes superiores a los líderes en volumen.

In [ ]:
# Sección 2: Contribución de Ingresos y Margen por Canal de Venta

# Agregación de datos por canal
contrib_canal = df_op.groupby('CANAL').agg({
    'Neto': 'sum'
})

## 3. Rentabilidad versus Volumen: Análisis de Mix de Productos

En este apartado evaluamos la relación entre el volumen de ventas (cantidad de unidades) y la rentabilidad (margen bruto) por producto para identificar oportunidades de optimización del mix. Este análisis permite detectar productos con alto margen pero bajo volumen (nichos premium) y productos con alta penetración pero márgenes comprimidos, orientando decisiones de pricing y promoción diferenciadas por segmento.

In [ ]:
# SECCIÓN 3: RENTABILIDAD VERSUS VOLUMEN - ANÁLISIS DE MIX DE PRODUCTOS

# Preparar datos a nivel de producto
df_mix = df_op.groupby(['DESCRIPCION', 'GRUPO']).agg({
    'CANTIDAD': 'sum'
}).reset_index()

## 4. Desempeño de KAMs y Gestión de Cartera

Esta sección analiza la productividad individual de cada Key Account Manager (KAM) y su impacto en la retención y crecimiento de clientes. Se evaluará la dispersión de ventas entre gestores y la relación entre diversificación de cartera y retención de clientes, identificando patrones de desempeño que permitan optimizar la asignación de recursos y la estrategia de gestión comercial.

In [ ]:
# Sección 4: Desempeño de KAMs y Gestión de Cartera
# Pregunta 3: ¿Cuál es el desempeño de cada KAM y cómo impacta en la retención y crecimiento de clientes?

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

# ===== ANÁLISIS 1: VENTAS POR KAM =====
df_kam_ventas = df_op.groupby('KAM').agg({
    'VENTA NETA23': 'sum'
}).reset_index()

## 5. Concentración de Clientes y Riesgo de Dependencia

Ecovida mantiene una alta concentración de ventas en un reducido grupo de clientes mayoristas, siendo Walmart y Cencosud los principales canales de distribución. Esta concentración representa un riesgo estratégico significativo que requiere diversificación urgente hacia canales digitales y mayoristas independientes para asegurar la sostenibilidad y crecimiento del negocio.

In [ ]:
# Sección 5: Concentración de Clientes y Riesgo de Dependencia

# 5.1 Análisis de concentración por cliente (RAZON SOCIAL)
concentracion_clientes = df_op.groupby('RAZON SOCIAL').agg({
    'Neto': 'sum'
}).reset_index()

## 6. Recomendaciones Estratégicas y Plan de Acción

Esta sección analiza la matriz de oportunidades cruzando CANAL y KAM para identificar combinaciones con mayor potencial de crecimiento en margen y ventas netas. Se segmentan iniciativas en quick-wins (digital, expansión de mayoristas) versus transformacionales (reposicionamiento de marca) con impacto financiero estimado, dirigidas a optimizar la estrategia comercial de Ecovida.

In [ ]:
# ===========================
# SECCIÓN 6: RECOMENDACIONES ESTRATÉGICAS Y PLAN DE ACCIÓN
# ===========================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- 6.1 MATRIZ DE OPORTUNIDADES: CANAL x KAM ---
# Crear tabla pivote con Margen_Bruto_23 y VENTA NETA23
matriz_margen = df_op.groupby(['CANAL', 'KAM']).agg({
    'Margen_Bruto_23': 'mean'
}).reset_index()

---

## Conclusiones

Este análisis respondió las siguientes preguntas de negocio:

- ¿Cuál es la contribución de ingresos y margen por canal de venta y cómo evolucionó en el período?
- ¿Qué productos generan mayor rentabilidad versus volumen y existen oportunidades de mix de productos?
- ¿Cuál es el desempeño de cada KAM y cómo impacta en la retención y crecimiento de clientes?
- ¿Cómo optimizar la estrategia comercial considerando la alta concentración en Walmart y Cencosud?

---
*Análisis generado con Python · pandas · matplotlib · seaborn*
*Dataset: 192,831 transacciones · 2023-01-01 → 2025-01-14*